In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# ==========================================
# 1. Load Data
# ==========================================
print("Loading dataset...")
df = pd.read_csv("csic_database.csv") # Yahan apni file ka path dalo

# ==========================================
# 2. Drop the Trash (Noise)
# ==========================================
cols_to_drop = [
    'Unnamed: 0', 'Pragma', 'Cache-Control', 'Accept', 
    'Accept-encoding', 'Accept-charset', 'language', 'host'
]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# ==========================================
# 3. Handle Missing Values
# ==========================================
print("Handling missing values...")
# Text columns: Replace NaN with empty string
text_cols = ['URL', 'content', 'cookie', 'User-Agent']
for col in text_cols:
    df[col] = df[col].fillna("")

# Categorical columns: Replace NaN with 'unknown'
cat_cols = ['Method', 'content-type', 'connection']
for col in cat_cols:
    df[col] = df[col].fillna("unknown")

# Numerical columns: Force to number, replace NaN with 0
df['lenght'] = pd.to_numeric(df['lenght'], errors='coerce').fillna(0)

# ==========================================
# 4. Feature Engineering (The Master Payload)
# ==========================================
# Alag-alag text columns ko TF-IDF dene se better hai sabko ek me merge kar do
df['Master_Payload'] = df['URL'] + " " + df['content'] + " " + df['cookie'] + " " + df['User-Agent']

# ==========================================
# 5. Define X (Features) and y (Target)
# ==========================================
X = df[['Master_Payload', 'Method', 'content-type', 'connection', 'lenght']]

# Target variable ko 0 aur 1 me convert karo
le = LabelEncoder()
y = le.fit_transform(df['classification'])
print(f"Classes mapped as: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ==========================================
# 6. Build the Preprocessing Engine (ColumnTransformer)
# ==========================================
# Yeh engine decide karega kis column ke saath kya karna hai
preprocessor = ColumnTransformer(
    transformers=[
        # Text par Character-level TF-IDF lagao (Cyber attacks pakadne ke liye best)
        ('text_tf', TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 3), max_features=5000), 'Master_Payload'),
        
        # Categorical par One-Hot Encoding lagao
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore'), ['Method', 'content-type', 'connection']),
        
        # Numerical par Scaling lagao
        ('num_scale', StandardScaler(), ['lenght'])
    ]
)

# ==========================================
# 7. Create Pipeline and Train
# ==========================================
# Pipeline = Preprocessor + Model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
])

print("Training the model... (Yeh thoda time lega)")
pipeline.fit(X_train, y_train)

# ==========================================
# 8. Evaluate the Model
# ==========================================
print("Evaluating model on Test Data...")
y_pred = pipeline.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ==========================================
# 9. Save for FastAPI Backend
# ==========================================
model_filename = 'deeplog_pipeline.pkl'
joblib.dump(pipeline, model_filename)
print(f"\nModel aur Preprocessor successfully save ho gaye '{model_filename}' me.")

Loading dataset...
Handling missing values...
Classes mapped as: {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}
Training the model... (Yeh thoda time lega)
Evaluating model on Test Data...

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99      7200
           1       0.99      0.97      0.98      5013

    accuracy                           0.98     12213
   macro avg       0.98      0.98      0.98     12213
weighted avg       0.98      0.98      0.98     12213


Model aur Preprocessor successfully save ho gaye 'deeplog_pipeline.pkl' me.
